In [1]:
! pip install pandas numpy torch biopython requests fastparquet

In [2]:
from Bio.PDB import *
import sys
from pathlib import Path

parser = MMCIFParser()


nonmetals = ['C', 'N', 'O', 'S', 'P']
metals = ['Zn', 'Co', 'Mg', 'Fe', 'Cu', 'Mn', 'Mo', 'Ni']
fnames = [f.name for f in Path('./data/cifs').iterdir() if f.is_file()]
fpaths = [f'./data/cifs/{name}' for name in fnames]
#len(files)
#for f in files: 
#


In [3]:
import time
import pandas as pd
timed = False
chunk = []
count = 0
i = 0
for fn,fp in zip(fnames, fpaths):

    if timed:
        start = time.time_ns()
    metal = None
    structure = parser.get_structure(fn, fp)
    atoms = np.fromiter(structure.get_atoms(), object)
    elements = np.array([atom.element for atom in atoms])
    unique_e = np.unique(elements)
    coords = np.array([atom.coord for atom in atoms])

    metal_mask = np.isin(elements, metals)  # use isin against fixed list
    if not metal_mask.any():
        continue
    rows = np.column_stack([np.full(len(atoms), fn), metal_mask, elements, coords])
    chunk.append(rows)
    if len(chunk) >= 1000:
        pd.DataFrame(np.vstack(chunk), columns=['structure_id', 'is_metal', 'element', 'x', 'y', 'z']).to_parquet(f'data/chunk_{i}.parquet')        
        chunk = []
        i += 1
    if timed:
        finish = time.time_ns()
        print(f"{(finish-start)/10**9}")
        break
    count += 1
    if count % 99 == 0:
        print(f'\n\t{count} ITERATIONS')


#df = pd.DataFrame(np.array(data).reshape(-1,6))
#df


/Users/stewartgreen/CODE/CS540/Final Project/.venv/lib/python3.13/site-packages/Bio/PDB/StructureBuilder.py:100: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 8585.
  warnings.warn(
/Users/stewartgreen/CODE/CS540/Final Project/.venv/lib/python3.13/site-packages/Bio/PDB/StructureBuilder.py:100: PDBConstructionWarning: WARNING: Chain AAA is discontinuous at line 4442.
  warnings.warn(
/Users/stewartgreen/CODE/CS540/Final Project/.venv/lib/python3.13/site-packages/Bio/PDB/StructureBuilder.py:100: PDBConstructionWarning: WARNING: Chain BBB is discontinuous at line 4479.
  warnings.warn(
/Users/stewartgreen/CODE/CS540/Final Project/.venv/lib/python3.13/site-packages/Bio/PDB/StructureBuilder.py:100: PDBConstructionWarning: WARNING: Chain AAA is discontinuous at line 4523.
  warnings.warn(
/Users/stewartgreen/CODE/CS540/Final Project/.venv/lib/python3.13/site-packages/Bio/PDB/StructureBuilder.py:100: PDBConstructionWarning: WARNING: Chain BBB is discontinuous at line 4578.

KeyboardInterrupt: 

In [ ]:
import gemmi
import pandas as pd
import numpy as np

BACKBONE = {'N', 'CA', 'C', 'O', 'OXT'}
COLS = ['structure_id', 'chain', 'res_name', 'res_num',
        'ca_x', 'ca_y', 'ca_z',
        'n_x',  'n_y',  'n_z',
        'o_x',  'o_y',  'o_z',
        'sc_x', 'sc_y', 'sc_z',
        'is_metal']

chunk = []
i = 0

def pos(residue, name):
    a = residue.find_atom(name, '*')
    return [a.pos.x, a.pos.y, a.pos.z] if a else [np.nan, np.nan, np.nan]

for fn, fp in zip(fnames, fpaths):
    try:
        model = gemmi.read_structure(fp)[0]
        rows = []

        for chain in model:
            for residue in chain:
                res_info = gemmi.find_tabulated_residue(residue.name)
                is_metal = len(residue) > 0 and residue[0].element.name in metals

                if res_info.is_water():
                    continue
                if not res_info.is_amino_acid() and not is_metal:
                    continue

                sc = [a for a in residue if a.name not in BACKBONE]
                sc_pos = np.mean([[a.pos.x, a.pos.y, a.pos.z] for a in sc], axis=0).tolist() if sc else [np.nan, np.nan, np.nan]

                rows.append([fn, chain.name, residue.name, str(residue.seqid),
                             *pos(residue, 'CA'), *pos(residue, 'N'), *pos(residue, 'O'), *sc_pos, is_metal])

        chunk.extend(rows)
        if len(chunk) >= 50_000:
            pd.DataFrame(chunk, columns=COLS).to_parquet(f'data/gemmi/chunk_{i}.parquet')
            chunk = []
            i += 1
    except Exception:
        continue

if chunk:
    pd.DataFrame(chunk, columns=COLS).to_parquet(f'data/gemmi/chunk_{i}.parquet')

In [ ]:
import gemmi

with open('data/sequences.fasta', 'w') as f:
    for fn, fp in zip(fnames, fpaths):
        try:
            st = gemmi.read_structure(fp)
            for chain in st[0]:
                polymer = chain.get_polymer()
                if polymer.check_polymer_type() in (gemmi.PolymerType.PeptideL, gemmi.PolymerType.PeptideD):
                    seq = polymer.make_one_letter_sequence()
                    f.write(f'>{fn}\n{seq}\n')
                    break
        except:
            continue

In [ ]:
! mmseqs easy-cluster data/sequences.fasta data/clusters data/tmp --min-seq-id 0.3 -c 0.8

In [ ]:
import gemmi
import numpy as np
import pandas as pd
import os

os.makedirs('data/gemmi_atom14', exist_ok=True)

ATOM14_NAMES = {
    'ALA': ['N', 'CA', 'C', 'O', 'CB', '', '', '', '', '', '', '', '', ''],
    'ARG': ['N', 'CA', 'C', 'O', 'CB', 'CG', 'CD', 'NE', 'CZ', 'NH1', 'NH2', '', '', ''],
    'ASN': ['N', 'CA', 'C', 'O', 'CB', 'CG', 'OD1', 'ND2', '', '', '', '', '', ''],
    'ASP': ['N', 'CA', 'C', 'O', 'CB', 'CG', 'OD1', 'OD2', '', '', '', '', '', ''],
    'CYS': ['N', 'CA', 'C', 'O', 'CB', 'SG', '', '', '', '', '', '', '', ''],
    'GLN': ['N', 'CA', 'C', 'O', 'CB', 'CG', 'CD', 'OE1', 'NE2', '', '', '', '', ''],
    'GLU': ['N', 'CA', 'C', 'O', 'CB', 'CG', 'CD', 'OE1', 'OE2', '', '', '', '', ''],
    'GLY': ['N', 'CA', 'C', 'O', '', '', '', '', '', '', '', '', '', ''],
    'HIS': ['N', 'CA', 'C', 'O', 'CB', 'CG', 'ND1', 'CD2', 'CE1', 'NE2', '', '', '', ''],
    'ILE': ['N', 'CA', 'C', 'O', 'CB', 'CG1', 'CG2', 'CD1', '', '', '', '', '', ''],
    'LEU': ['N', 'CA', 'C', 'O', 'CB', 'CG', 'CD1', 'CD2', '', '', '', '', '', ''],
    'LYS': ['N', 'CA', 'C', 'O', 'CB', 'CG', 'CD', 'CE', 'NZ', '', '', '', '', ''],
    'MET': ['N', 'CA', 'C', 'O', 'CB', 'CG', 'SD', 'CE', '', '', '', '', '', ''],
    'PHE': ['N', 'CA', 'C', 'O', 'CB', 'CG', 'CD1', 'CD2', 'CE1', 'CE2', 'CZ', '', '', ''],
    'PRO': ['N', 'CA', 'C', 'O', 'CB', 'CG', 'CD', '', '', '', '', '', '', ''],
    'SER': ['N', 'CA', 'C', 'O', 'CB', 'OG', '', '', '', '', '', '', '', ''],
    'THR': ['N', 'CA', 'C', 'O', 'CB', 'OG1', 'CG2', '', '', '', '', '', '', ''],
    'TRP': ['N', 'CA', 'C', 'O', 'CB', 'CG', 'CD1', 'CD2', 'NE1', 'CE2', 'CE3', 'CZ2', 'CZ3', 'CH2'],
    'TYR': ['N', 'CA', 'C', 'O', 'CB', 'CG', 'CD1', 'CD2', 'CE1', 'CE2', 'CZ', 'OH', '', ''],
    'VAL': ['N', 'CA', 'C', 'O', 'CB', 'CG1', 'CG2', '', '', '', '', '', '', ''],
}

COORD_COLS = [f'{c}{i}' for i in range(14) for c in ('x', 'y', 'z')]
COLS = ['structure_id', 'chain', 'res_name', 'res_num', 'is_metal'] + COORD_COLS

chunk = []
i = 0

for fn, fp in zip(fnames, fpaths):
    try:
        model = gemmi.read_structure(fp)[0]
        rows = []

        for chain in model:
            for residue in chain:
                res_info = gemmi.find_tabulated_residue(residue.name)
                is_metal = len(residue) > 0 and residue[0].element.name in metals

                if res_info.is_water():
                    continue
                if not res_info.is_amino_acid() and not is_metal:
                    continue

                coords = [np.nan] * 42  # 14 slots * 3

                if is_metal:
                    a = residue[0]
                    coords[0], coords[1], coords[2] = a.pos.x, a.pos.y, a.pos.z
                else:
                    atom14 = ATOM14_NAMES.get(residue.name)
                    if atom14 is None:
                        continue  # non-standard residue
                    for slot, atom_name in enumerate(atom14):
                        if not atom_name:
                            continue
                        a = residue.find_atom(atom_name, '*')
                        if a:
                            coords[slot*3]     = a.pos.x
                            coords[slot*3 + 1] = a.pos.y
                            coords[slot*3 + 2] = a.pos.z

                rows.append([fn, chain.name, residue.name, str(residue.seqid), is_metal] + coords)

        chunk.extend(rows)
        if len(chunk) >= 50_000:
            pd.DataFrame(chunk, columns=COLS).to_parquet(f'data/gemmi_atom14/chunk_{i}.parquet')
            chunk = []
            i += 1
    except Exception:
        continue

if chunk:
    pd.DataFrame(chunk, columns=COLS).to_parquet(f'data/gemmi_atom14/chunk_{i}.parquet')

In [ ]:
import pandas as pd
import glob

clusters = pd.read_csv('data/clusters_cluster.tsv', sep='\t', names=['rep', 'member'])

chunks = [pd.read_parquet(f) for f in glob.glob('data/gemmi_atom14/chunk_*.parquet')]
df = pd.concat(chunks)

# Add cluster_id column (the representative for each structure's cluster)
member_to_cluster = dict(zip(clusters['member'], clusters['rep']))
df['cluster_id'] = df['structure_id'].map(member_to_cluster)

df.to_parquet('data/clusters/clusters.parquet')
print(f'{df["structure_id"].nunique()} structures across {df["cluster_id"].nunique()} clusters')

In [ ]:
import gemmi
import torch
import glob
import os

CACHE_DIR   = 'data/pt_cache_v2'
CIF_DIR     = 'data/cifs'
ADD_METALS  = {'Ca', 'Cd', 'V', 'W'}   # elements missing from original parse

# Find zero-metal cached structures
zero_metal_ids = []
for f in glob.glob(f'{CACHE_DIR}/*.pt'):
    d = torch.load(f, weights_only=False)
    if d.y.shape[0] == 0:
        zero_metal_ids.append(os.path.basename(f).replace('.pt', ''))

print(f'Zero-metal cached structures to patch: {len(zero_metal_ids)}')

patched    = 0
no_metal   = 0
no_cif     = 0
errors     = 0
found_by   = {el: 0 for el in ADD_METALS}

for sid in zero_metal_ids:
    cif_path = os.path.join(CIF_DIR, sid)
    if not os.path.exists(cif_path):
        no_cif += 1
        continue

    try:
        model = gemmi.read_structure(cif_path)[0]
        new_coords = []
        for chain in model:
            for residue in chain:
                if len(residue) > 0 and residue[0].element.name in ADD_METALS:
                    a = residue[0]
                    new_coords.append([a.pos.x, a.pos.y, a.pos.z])
                    found_by[residue[0].element.name] = found_by.get(residue[0].element.name, 0) + 1

        if not new_coords:
            no_metal += 1
            continue

        cache_path = os.path.join(CACHE_DIR, f'{sid}.pt')
        d = torch.load(cache_path, weights_only=False)
        d.y = torch.tensor(new_coords, dtype=torch.float)
        torch.save(d, cache_path)
        patched += 1

    except Exception:
        errors += 1
        continue

    if patched % 500 == 0 and patched > 0:
        print(f'  patched {patched}...')

print(f'\nDone.')
print(f'  Patched:         {patched}')
print(f'  No target metal: {no_metal}')
print(f'  CIF not on disk: {no_cif}')
print(f'  Errors:          {errors}')
print(f'\n  Metal sites added:')
for el, count in found_by.items():
    print(f'    {el}: {count}')


In [ ]:
import gemmi
import torch
import glob
import os
from collections import Counter

CACHE_DIR = 'data/pt_cache_v2'
CIF_DIR   = 'data/cifs'

# Find still-zero-metal structures after the patch
zero_metal_ids = []
for f in glob.glob(f'{CACHE_DIR}/*.pt'):
    d = torch.load(f, weights_only=False)
    if d.y.shape[0] == 0:
        zero_metal_ids.append(os.path.basename(f).replace('.pt', ''))

print(f'Remaining zero-metal structures: {len(zero_metal_ids)}')

element_counts  = Counter()
residue_counts  = Counter()

for sid in zero_metal_ids:
    cif_path = os.path.join(CIF_DIR, sid)
    if not os.path.exists(cif_path):
        continue
    try:
        model = gemmi.read_structure(cif_path)[0]
        for chain in model:
            for residue in chain:
                if residue.het_flag != 'H' or len(residue) == 0:
                    continue
                if gemmi.find_tabulated_residue(residue.name).is_water():
                    continue
                el = residue[0].element.name
                element_counts[el] += 1
                residue_counts[residue.name] += 1
    except Exception:
        continue

print('\nTop 30 heteroatom elements in remaining zero-metal structures:')
for el, count in element_counts.most_common(30):
    print(f'  {el:4s}  {count:6d}')

print('\nTop 20 heteroatom residue names:')
for name, count in residue_counts.most_common(20):
    print(f'  {name:6s}  {count:6d}')


/Users/stewartgreen/CODE/CS540/Final Project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Remaining zero-metal structures: 7615

Top 30 heteroatom elements in remaining zero-metal structures:
  C      27224
  Na      9227
  N       7441
  Cl      6075
  S       5543
  P       3332
  O       2746
  K       2506
  I        488
  Hg       476
  Pt       352
  Cs       351
  Br       299
  X        278
  Ru       129
  Xe       108
  Au       105
  Rh       104
  F        102
  Tb        81
  Sr        75
  As        73
  Gd        73
  Ag        73
  Rb        72
  Yb        59
  Sm        55
  La        54
  Li        51
  Y         50

Top 20 heteroatom residue names:
  NA        9227
  CL        6029
  EDO       5118
  GOL       4711
  HEM       3886
  SO4       3507
  MSE       2526
  K         2506
  NAG       1727
  DMS       1639
  PO4       1603
  ACT       1351
  PEG        850
  MPD        737
  FMT        677
  NAD        525
  IOD        484
  NO3        438
  HG         436
  CS         351
